In [2]:
import asyncio
import nest_asyncio
import logging
import pandas as pd
import pyotp 
from collections import defaultdict, deque
from threading import Thread
from threading import Lock
from datetime import datetime
import numba_indicators
resampled_data = {}
last_resample_time = {}

from NorenRestApiPy.NorenApi import NorenApi
# Initialize logger
logger = logging.getLogger('ShoonyaApi')
logger.setLevel(logging.INFO) 
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

# Global variables
current_positions = {}
feed_opened = False
feed_lock = Lock()
position_lock = asyncio.Lock()
feedJson = defaultdict(lambda: deque(maxlen=20))
extra_feedJson = defaultdict(lambda: deque(maxlen=20))
api = None
exchange = 'NSE'
Initial_token = '26009'
subscription_string = f"{exchange}|{Initial_token}"
resample_frequency = "5s"
symbol = "BANKNIFTY"

class TradingState:
    def __init__(self):
        self.ce_trading_symbol = None
        self.pe_trading_symbol = None
        self.ce_trading_token = None
        self.pe_trading_token = None

state = TradingState()

def initialize_api(credentials_file="usercred.xlsx"):
    global api
    api = NorenApi(
        host="https://api.shoonya.com/NorenWClientTP/",
        websocket="wss://api.shoonya.com/NorenWSTP/"
    )

    credentials = pd.read_excel(credentials_file)
    user = credentials.iloc[0, 0]
    password = credentials.iloc[0, 1]
    vendor_code = credentials.iloc[0, 2]
    app_key = credentials.iloc[0, 3]
    imei = credentials.iloc[0, 4]
    qr_code = credentials.iloc[0, 5]
    factor2 = pyotp.TOTP(qr_code).now()

    api.login_result = api.login(
        userid=user,
        password=password,
        twoFA=factor2,
        vendor_code=vendor_code,
        api_secret=app_key,
        imei=imei
    )


async def connect_and_subscribe():
    global feed_opened
    retry_delay = 1  # Initial retry delay in seconds
    max_retry_delay = 32  # Maximum retry delay in seconds
    while True:
        try:
            api.start_websocket(
                order_update_callback=event_handler_order_update,
                subscribe_callback=event_handler_feed_update,
                socket_open_callback=open_callback
            )
            while not feed_opened:
                await asyncio.sleep(1)
            api.subscribe([subscription_string])
            logger.info("WebSocket connected and subscribed successfully.")
            retry_delay = 1  # Reset retry delay after successful connection
            break  # Exit the loop if connection is successful
        except Exception as e:
            logger.error(f"WebSocket connection error: {e}")
            logger.info(f"Reconnecting in {retry_delay} seconds...")
            await asyncio.sleep(retry_delay)
            retry_delay = min(retry_delay * 2, max_retry_delay)  # Exponential backoff

def event_handler_order_update(data):
    logger.info(f"Order update: {data}")

def open_callback():
    global feed_opened
    if not feed_opened:
        feed_opened = True
        logger.info('Feed Opened')
    else:
        logger.warning('Feed Opened callback called multiple times.')

def event_handler_feed_update(tick_data):
    #print(tick_data)
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with feed_lock:  # Acquire lock for thread-safety

                    if token == Initial_token:
                        feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
                    else:
                        extra_feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})

    except (KeyError, ValueError) as e:
        logger.error(f"Error processing tick data: {e}")

async def resample_ticks():
    global resampled_data
    
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with feed_lock:
            temp_resampled_data = {}
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            # Initialize the DataFrame with the first resample
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()
                            temp_resampled_data[token] = df_resampled
                            last_resample_time[token] = df_resampled.index.max()
                        else:
                            # Create a copy of the existing data
                            temp_df = resampled_data[token].copy()
                            
                            # Resample the new ticks
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                            # Update the temporary DataFrame with new data
                            for idx, row in df_resampled.iterrows():
                                if idx in temp_df.index:
                                    temp_df.loc[idx, 'high'] = max(temp_df.loc[idx, 'high'], row['high'])
                                    temp_df.loc[idx, 'low'] = min(temp_df.loc[idx, 'low'], row['low'])
                                    temp_df.loc[idx, 'close'] = row['close']
                                else:
                                    temp_df.loc[idx] = row

                            # Update the last resampled time
                            last_resample_time[token] = df_resampled.index.max()
                            
                            temp_resampled_data[token] = temp_df

                        if token in temp_resampled_data:
                            long_stop, short_stop, direction = numba_indicators.chandelier_exit_numba(
                                temp_resampled_data[token]['open'].values,
                                temp_resampled_data[token]['high'].values,
                                temp_resampled_data[token]['low'].values,
                                temp_resampled_data[token]['close'].values
                            )

                            temp_resampled_data[token]['long_stop'] = long_stop
                            temp_resampled_data[token]['short_stop'] = short_stop
                            temp_resampled_data[token]['ce_direction'] = direction

                        # Clean up the data
                        temp_resampled_data[token] = temp_resampled_data[token].dropna(subset=['open', 'high', 'low', 'close'])

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

            # After processing all tokens, update the global resampled_data
            resampled_data = temp_resampled_data

        await asyncio.sleep(0.1)

async def main():

    initialize_api()
    await connect_and_subscribe()
    resample_task = asyncio.create_task(resample_ticks())
    lot_size_task = asyncio.create_task(get_lotsize()) 
    candle_end_finder_task = asyncio.create_task(candle_end_finder())     
    await asyncio.gather(lot_size_task, resample_task, candle_end_finder_task)

loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()

# Always create a task for 'main' within the current loop
asyncio.create_task(main())

# If the loop wasn't running, start it
if not loop.is_running():
    loop.run_forever() 

2024-08-06 04:47:48,199 - ShoonyaApi - INFO - Feed Opened
2024-08-06 04:47:49,092 - ShoonyaApi - INFO - WebSocket connected and subscribed successfully.
2024-08-06 04:47:49,095 - ShoonyaApi - INFO - Starting process_token_data
2024-08-06 04:47:50,004 - ShoonyaApi - WARNING - resampled_data is empty.
2024-08-06 04:47:55,002 - ShoonyaApi - WARNING - Not enough data or token 26009 not found. Need at least 3 candles.
2024-08-06 04:48:05,002 - ShoonyaApi - WARNING - Not enough data or token 26009 not found. Need at least 3 candles.


In [9]:
resampled_data

{'26009':                         open     high      low    close  long_stop  \
 tt                                                                   
 2024-08-06 04:45:05  50092.1  50092.1  50092.1  50092.1        NaN   
 
                      short_stop  ce_direction  
 tt                                             
 2024-08-06 04:45:05         NaN           0.0  }

In [4]:
import pandas as pd
# Load the CSV files into DataFrames
nfo_scrips_df = pd.read_csv('/home/deep/Desktop/NEWshoonya/NFO_symbols.csv')
mcx_scrips_df = pd.read_csv('/home/deep/Desktop/NEWshoonya/MCX_symbols.csv')

async def get_lotsize():
     
    while True:
        await asyncio.sleep(5)
        
        if exchange.upper() == 'NSE':
            df = nfo_scrips_df
        elif exchange.upper() == 'MCX':
            df = mcx_scrips_df
        else:
            raise ValueError("Unsupported exchange. Please use 'NSE' or 'MCX'.")

        # Find the first matching symbol
        match = df[df['Symbol'] == symbol]
        
        if not match.empty:
            # Get LotSize
            lotsize = match.iloc[0]['LotSize']
            
            # Check feedJson if token is provided
            if Initial_token:
                if not feedJson[Initial_token]:
                    logger.info(f"FeedJson for token {Initial_token} is empty. LotSize: {lotsize}.")                    
                else:
                    # Retrieve the last recent price
                    last_price = int(feedJson[Initial_token][-1]['ltp'])
                    # Calculate ATM strike price
                    mod = int(last_price) % 100
                    atm_strike = last_price - mod if mod <= 50 else last_price + (100 - mod)                    
                    # Find trading symbols for ATM strike
                    filtered_df = df[
                        (df['Symbol'] == symbol) &
                        (df['StrikePrice'] == float(atm_strike))
                    ]
                    
                    if filtered_df.empty:
                        logger.info(f"Could not find trading symbols for ATM strike {atm_strike}")
                        #return lotsize, last_price, None, None
                        logger.info(f"wait for lot size to update {lotsize}.")

                    # Find the CE and PE trading symbols
                    ce_row = filtered_df[filtered_df['OptionType'] == 'CE'].sort_values('Expiry').iloc[0]
                    pe_row = filtered_df[filtered_df['OptionType'] == 'PE'].sort_values('Expiry').iloc[0]

                    ce_trading_symbol, pe_trading_symbol = ce_row['TradingSymbol'], pe_row['TradingSymbol']
                    ce_trading_token, pe_trading_token = ce_row['Token'], pe_row['Token']                    
                    
                      # Update the state
                    state.ce_trading_symbol = ce_trading_symbol
                    state.pe_trading_symbol = pe_trading_symbol
                    state.ce_trading_token = ce_trading_token
                    state.pe_trading_token = pe_trading_token

                    #logger.info(f"CE Symbol: {ce_trading_symbol}, PE Symbol: {pe_trading_symbol}")            
        else:
            logger.warning(f"{symbol} not found on {exchange}.")

In [5]:
# async def candle_end_finder():
#     global resampled_data, resample_frequency
#     logger.info("Starting process_token_data")

#     while True:
        
#         current_time = pd.Timestamp.now()
#         time_bucket_start = current_time.floor(resample_frequency)
#         time_bucket_end = time_bucket_start + pd.Timedelta(resample_frequency)
#         wait_time = (time_bucket_end - current_time).total_seconds() + 0.001

#         if wait_time > 0:
#             #logger.info(f"Waiting for {wait_time:.2f} seconds until the end of the current time bucket")
#             await asyncio.sleep(wait_time)
        
#         with feed_lock:
#            # logger.info("Acquired lock and accessing resampled_data")
#             if not resampled_data:
#                 logger.warning("resampled_data is empty.")
#             else:
#                 for token, df in resampled_data.items():
#                     if df is None or df.empty:
#                         logger.warning(f"DataFrame for token {token} is empty or None.")
#                         continue

#                     if len(df) < 3:
#                         logger.warning(f"Not enough data or token {token} not found. Need at least 3 candles.")
#                         continue
                                                
#                     print(f"[{pd.Timestamp.now()}] Processing data for token: {token}")

#                     if token not in last_processed_candle or last_processed_candle[token] < time_bucket_end:
                         
#                         try:
#                             if time_bucket_start == df.index[-1]:
#                                 completed_candles_df = df
#                                 last_processed_candle[token] = time_bucket_start

#                             elif time_bucket_end == df.index[-1]:
#                                 completed_candles_df = df.loc[:time_bucket_end - pd.Timedelta(microseconds=1)]
#                                 last_processed_candle[token] = time_bucket_start
#                             else:
#                                 logger.warning(f"No new candle forming now {token}")
#                                 continue  # Skip to the next token

#                             logger.info(f"Completed candles DataFrame for token {token}: {completed_candles_df}")

#                             try:
#                                 required_columns = ['ce_direction']
#                                 missing_columns = [col for col in required_columns if col not in completed_candles_df.columns]
#                                 if missing_columns:
#                                     raise KeyError(f"Missing columns {missing_columns}")

#                                 current_direction = completed_candles_df['ce_direction'].iloc[-1]
#                                 previous_direction = completed_candles_df['ce_direction'].iloc[-2]

#                                 #await execute_trade_logic(token, current_direction, previous_direction, completed_candles_df)

#                             except KeyError as e:
#                                 logger.error(f"Error processing token {token}: {e}")

#                         except KeyError as e:
#                             logger.error(f"Error processing token {token}: Column {e} not found.")
#                         except IndexError:
#                             logger.error(f"Error processing token {token}: Not enough candles to calculate indicators.")
    
#         # # Calculate the time for the next bucket
#         # next_bucket_start = time_bucket_end
#         # next_bucket_end = next_bucket_start + pd.Timedelta(resample_frequency)
        
#         # # Calculate wait time for the next bucket
#         # wait_time = (next_bucket_end - pd.Timestamp.now()).total_seconds()
#         if wait_time > 0:
#             await asyncio.sleep(wait_time)


In [6]:
async def candle_end_finder():
    global resampled_data, resample_frequency
    logger.info("Starting process_token_data")

    while True:
        current_time = pd.Timestamp.now()
        time_bucket_start = current_time.floor(resample_frequency)
        time_bucket_end = time_bucket_start + pd.Timedelta(resample_frequency)
        wait_time = (time_bucket_end - current_time).total_seconds() + 0.001

        if wait_time > 0:
            await asyncio.sleep(wait_time)

        # Access the shared resource resampled_data inside the locked section
        data_copy = {}
        with feed_lock:
            if not resampled_data:
                logger.warning("resampled_data is empty.")
                continue

            for token, df in resampled_data.items():
                if df is None or df.empty:
                    logger.warning(f"DataFrame for token {token} is empty or None.")
                    continue

                if len(df) < 3:
                    logger.warning(f"Not enough data or token {token} not found. Need at least 3 candles.")
                    continue

                # Make a shallow copy of the relevant data
                data_copy[token] = df.copy()

        # Release the lock here as data_copy now contains a local copy of the data

        # Process the copied data outside the lock
        for token, df in data_copy.items():
            print(f"[{pd.Timestamp.now()}] Processing data for token: {token}")

            if token not in last_processed_candle or last_processed_candle[token] < time_bucket_end:
                try:
                    if time_bucket_start == df.index[-1]:
                        completed_candles_df = df
                        last_processed_candle[token] = time_bucket_start

                    elif time_bucket_end == df.index[-1]:
                        completed_candles_df = df.loc[:time_bucket_end - pd.Timedelta(microseconds=1)]
                        last_processed_candle[token] = time_bucket_start
                    else:
                        logger.warning(f"No new candle forming now for token {token}")
                        continue

                    logger.info(f"Completed candles DataFrame for token {token}: {completed_candles_df}")

                    try:
                        required_columns = ['ce_direction']
                        missing_columns = [col for col in required_columns if col not in completed_candles_df.columns]
                        if missing_columns:
                            raise KeyError(f"Missing columns {missing_columns}")

                        current_direction = completed_candles_df['ce_direction'].iloc[-1]
                        previous_direction = completed_candles_df['ce_direction'].iloc[-2]

                        # Call the trade logic function
                        await execute_trade_logic(token, current_direction, previous_direction, completed_candles_df)

                    except KeyError as e:
                        logger.error(f"Error processing token {token}: {e}")

                except KeyError as e:
                    logger.error(f"Error processing token {token}: Column {e} not found.")
                except IndexError:
                    logger.error(f"Error processing token {token}: Not enough candles to calculate indicators.")

        # Calculate the time for the next bucket
        
        if wait_time > 0:
            await asyncio.sleep(wait_time)


In [7]:
async def execute_trade_logic(token, current_direction, previous_direction, completed_candles_df):
    global current_positions, position_lock
    print("i am execute_trade_logic")

    async with position_lock:  
        # Exit Logic (Prioritized)
        if token in current_positions:
            position_data = current_positions.get[token , {}]
            op_token = position_data.get["option_token"]
            op_symbol = position_data.get["symbol_name"]
            position_type = position_data.get["position_type"]

            if (position_type == 'call_buy' and current_direction == -1 and previous_direction == 1) or \
               (position_type == 'put_buy' and current_direction == 1 and previous_direction == -1):
                
                try:
                    await exit_trade(token, position_type, op_token, op_symbol, "Regular Exit")
                except Exception as e:
                    print(f"Error exiting trade for {token}: {e}")

        # Entry Logic 
        if token not in current_positions:  
            if current_direction == 1 and previous_direction == -1: 
                position_type = "call_buy"
            elif current_direction == -1 and previous_direction == 1:
                position_type = "put_buy"
            else:
                return  # No entry signal

            try:
                await enter_trade(token, position_type, "Regular Entry")       
            except Exception as e:
                print(f"Error entering trade for {token}: {e}")


In [8]:
async def enter_trade(token, position_type, entry_type):
    async with position_lock:  
       
        option_token = state.ce_trading_token if position_type == "call_buy" else state.pe_trading_token
        symbol_name = state.ce_trading_symbol if position_type == "call_buy" else state.pe_trading_symbol
        
        current_positions[token] = {
                    "position_type": position_type,
                    "option_token": option_token,
                    "symbol_name": symbol_name
                }
        action_time = pd.Timestamp.now()
        log_trade(symbol_name, "ENTRY", action_time, entry_type) 
        

async def exit_trade(token, position_type, op_token, op_symbol, exit_type):
    async with position_lock:

        #price = get_latest_price_option(op_token)
        action_time = pd.Timestamp.now()
        log_trade(op_symbol, "EXIT", action_time, exit_type)
        current_positions.pop(token, None)  # Remove the entire token entry
        